# ⚛️ MACE — Simulador Atómico Avanzado
**Rubro:** Nanotecnología y Materiales  
**Para qué:** Predice energía atómica con precisión cuántica  
**Demo:** Molécula de agua H₂O  
**Datos propios:** Archivos XYZ

---
### 🧒 ¿Qué es MACE?
MACE es como un cerebro artificial que aprendió química cuántica.
Predice la energía y fuerzas de cualquier molécula en segundos.

---
### 🎮 Las 7 celdas de este notebook:
| Celda | Hace | Sirve para |
| 1 | Instalar | Preparar herramientas |
| 2 | Descargar modelo | Bajar cerebro entrenado |
| 3 | Energia | Saber que tan estable es una molecula |
| 4 | Fuerzas | Ver hacia donde se mueve cada atomo |
| 5 | Optimizar | Encontrar la forma mas relajada |
| 6 | Dinamica molecular | Ver como vibran los atomos |
| 7 | Tus datos | Usar tus propias moleculas (.xyz) |

### 🔑 Variables que puedes editar:
- Molecula: cambia atomos y posiciones en Atoms(...)
- Modelo: small / medium / large en model_url
- Device: cpu o cuda (GPU)
- Precision: fmax (0.001=fino, 0.1=rapido)
- Pasos: steps (100=corto, 10000=real)

In [ ]:
# ============================================================
# CELDA 1: Instalar las herramientas
# ============================================================
!pip install mace-torch ase -q 2>&1 | tail -1

In [ ]:
# ============================================================
# CELDA 2: Descargar el cerebro entrenado (solo 1ra vez)
# ============================================================

import urllib.request
import os

# --- VARIABLE EDITABLE: cambia small/medium/large ---
model_url = "https://github.com/ACEsuit/mace-foundations/releases/download/mace_mp_0/2023-12-03-mace-128-L1_epoch-199.model"
model_file = "mace_mp_medium.model"

if not os.path.exists(model_file):
    print("Descargando modelo MACE-MP-0 (medium, 4.7M params)...")
    urllib.request.urlretrieve(model_url, model_file)
    print("Modelo descargado!")
else:
    print("Modelo ya descargado.")

In [ ]:
# ============================================================
# CELDA 3: ENERGIA - Crear molecula y calcular que tan estable es
# ============================================================
# Energia NEGATIVA = estable (atomos unidos)
# Energia POSITIVA = inestable (atomos se repelen)

from mace.calculators import MACECalculator
from ase import Atoms

# --- VARIABLES EDITABLES ---
# Cambia los atomos y posiciones:
# Ejemplos listos para copiar y pegar:
#   Metano: Atoms('CH4', positions=[[0,0,0],[0.6,0.6,0.6],[0.6,-0.6,-0.6],[-0.6,0.6,-0.6],[-0.6,-0.6,0.6]])
#   Sal:    Atoms('NaCl', positions=[[0,0,0],[2.8,0,0]])
#   Oro:    Atoms('Au2', positions=[[0,0,0],[2.88,0,0]])

water = Atoms('H2O', positions=[
    [0.00, 0.00, 0.00],   # Atomo 1: Oxigeno (O)
    [0.96, 0.00, 0.00],   # Atomo 2: Hidrogeno (H)
    [-0.24, 0.93, 0.00]   # Atomo 3: Hidrogeno (H)
])

water.calc = MACECalculator(model_paths=model_file, device='cpu')

energy = water.get_potential_energy()
print(f"Energia del agua: {energy:.4f} eV")
print("Negativo = estable | Positivo = inestable")

In [ ]:
# ============================================================
# CELDA 4: FUERZAS - Ver hacia donde se mueve cada atomo
# ============================================================
# Fuerzas en eV/Angstrom.
# Cercanas a 0 = atomo comodo donde esta.
# Lejos de 0 = atomo quiere moverse (necesitas optimizar).

forces = water.get_forces()
print("Fuerzas sobre cada atomo (eV/Angstrom):")
for i, (simbolo, f) in enumerate(zip(water.get_chemical_symbols(), forces)):
    print(f"  Atomo {i+1} ({simbolo}): fx={f[0]:+.4f}  fy={f[1]:+.4f}  fz={f[2]:+.4f}")

In [ ]:
# ============================================================
# CELDA 5: OPTIMIZAR - Encontrar la forma mas relajada
# ============================================================
# Mueve atomos poquito a poco hasta que las fuerzas sean ~0.
# Como una pelota que rueda hasta el fondo del valle.

from ase.optimize import BFGS

# --- VARIABLES EDITABLES ---
# fmax: 0.001=super preciso, 0.01=normal, 0.1=rapido

print("Buscando la forma mas estable...")
opt = BFGS(water)
opt.run(fmax=0.001, steps=200)

print(f"Energia optimizada: {water.get_potential_energy():.4f} eV")
print("Posiciones finales (Angstrom):")
for i, (s, p) in enumerate(zip(water.get_chemical_symbols(), water.positions)):
    print(f"  {s}: {p[0]:.4f}  {p[1]:.4f}  {p[2]:.4f}")

In [ ]:
# ============================================================
# CELDA 6: DINAMICA MOLECULAR - Simular movimiento
# ============================================================
# Atomos se mueven como si tuvieran temperatura.
# Como una pelicula de la molecula vibrando.

from ase.md.verlet import VelocityVerlet
from ase import units

water_md = Atoms('H2O', positions=[[0,0,0],[0.96,0,0],[-0.24,0.93,0]])
water_md.calc = MACECalculator(model_paths=model_file, device='cpu')

# --- VARIABLES EDITABLES ---
# timestep: 0.5=preciso, 1.0=normal, 2.0=rapido (en femtosegundos)
# num_steps: 100=corto, 1000=decente, 10000=real

timestep = 0.5 * units.fs
num_steps = 50

dyn = VelocityVerlet(water_md, timestep=timestep)
energies = []

print(f"Simulando {num_steps} pasos...")
for step in range(num_steps):
    dyn.run(1)
    e = water_md.get_potential_energy()
    energies.append(e)
    if step % 10 == 0:
        print(f"  Paso {step:3d}: energia = {e:.4f} eV")

print(f"Energia inicial: {energies[0]:.4f} eV")
print(f"Energia final:   {energies[-1]:.4f} eV")
print("(La energia varia = los atomos vibran)")

---
### 🧪 Como usar tus propios datos

Formato del archivo .xyz:
```
3                 <- numero de atomos
Mi molecula       <- comentario
O   0.00 0.00 0.00
H   0.96 0.00 0.00
H  -0.24 0.93 0.00
```
Sube el archivo a Colab (icono carpeta izquierda).

In [ ]:
# ============================================================
# CELDA 7: Cargar TU propia molecula desde .xyz
# ============================================================

from ase.io import read

# --- VARIABLE EDITABLE: cambia por el nombre de tu archivo ---
mi_molecula = read('mi_molecula.xyz')
mi_molecula.calc = MACECalculator(model_paths=model_file, device='cpu')

print(f"Atomos: {mi_molecula.get_chemical_symbols()}")
print(f"Energia: {mi_molecula.get_potential_energy():.4f} eV")
print("Ahora puedes usar las celdas 4, 5 y 6 con tu molecula!")